# 01 — Generate demo data
Writes a UC Delta table that both loader notebooks read in parallel.

In [0]:
%pip install -q faker
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("delta_staging_table", "genie_catalog.lakebase_source.lakebase_demo_events")
DELTA_STAGING_TABLE = dbutils.widgets.get("delta_staging_table")

ROW_COUNT = 100000

In [0]:
from datetime import timezone
import json, random
from faker import Faker
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, TimestampType,
)

fake = Faker()
Faker.seed(42)
random.seed(42)

EVENT_TYPES = ["click", "view", "purchase", "signup", "logout"]

def make_row(i: int) -> Row:
    return Row(
        event_id   = i,
        user_id    = random.randint(1, 1000),
        event_type = random.choice(EVENT_TYPES),
        payload    = json.dumps({
            "ip":      fake.ipv4(),
            "agent":   fake.user_agent(),
            "country": fake.country_code(),
        }),
        event_ts   = fake.date_time_between(
            start_date="-30d", end_date="now", tzinfo=timezone.utc
        ),
    )

schema = StructType([
    StructField("event_id",   LongType(),      False),
    StructField("user_id",    LongType(),      False),
    StructField("event_type", StringType(),    False),
    StructField("payload",    StringType(),    True),
    StructField("event_ts",   TimestampType(), False),
])

rows = [make_row(i) for i in range(1, ROW_COUNT + 1)]
df = spark.createDataFrame(rows, schema)

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(DELTA_STAGING_TABLE))

print(f"Wrote {df.count()} rows to {DELTA_STAGING_TABLE}")

Wrote 100000 rows to genie_catalog.lakebase_source.lakebase_demo_events
